In [82]:
"""
toar_rest_demo: demonstrate use the REST interface to TOAR database v2 from python

@author: s.schroeder, Forschungszentrum Juelich GmbH, Germany (s.schroeder@fz-juelich.de)
@date:   2025-22-07
"""
from io import StringIO
import json
import pandas as pd
import geopandas as gpd
import requests

token = 'eyJraWQiOiI1MTQ0NTM4ODMwNDI3Mjk1NzM2MjI2Mjc4MDM1NTUxMzg3NzY1NDMxNTY4OTIyMjAiLCJ0eXAiOiJhdCtqd3QiLCJhbGciOiJSUzI1NiJ9.eyJzdWIiOiJjNGEyODNjMC00ZjBhLTRjZTYtYjgxMy01ZjY5ZGNlNmU0ZjYiLCJhdWQiOiJ0b2FyX3NlcnZpY2VzIiwic2NvcGUiOiJkaXNwbGF5X25hbWUgb2ZmbGluZV9hY2Nlc3MgZW1haWwgcHJvZmlsZSBlZHVwZXJzb25fc2NvcGVkX2FmZmlsaWF0aW9uIGVkdXBlcnNvbl9lbnRpdGxlbWVudCBlZHVwZXJzb25fdW5pcXVlX2lkIGVkdXBlcnNvbl9hc3N1cmFuY2UiLCJhdXRoX3RpbWUiOjE3NjM4NjIzNDcsImlzcyI6Imh0dHBzOi8vbG9naW4uaGVsbWhvbHR6LmRlL29hdXRoMiIsImV4cCI6MTc2Mzg2NjczMCwiaWF0IjoxNzYzODYyNzMwLCJqdGkiOiI4ODI2MTU5NC0yNWNhLTRlOWUtODc4YS1kZDQ2ODMwM2U2MzAiLCJjbGllbnRfaWQiOiJ0b2FyX3NlcnZpY2VzIn0.TgyOGOPf6d5apZBnFiVTYG8Q2e_Lq5Ms0DpMuOMBze6QRBLk7sviETtpA4-NgExdwTvfFmjM3YtmHLu7oPtWlJVqX1DKt19kU47DKWu6EzGI_V1nKQYmNjwZo_b7OWnYyxFQKhnlHjYX90YL1Il63cxd4jUhNEDxPdn6a6fJFd8hL3MjKaOJBmb0O8GHaFVG6n5RvKd-d7Rvo8WhvQcRHNzEFbqCM6kvPF2nREPCXLeLNLxH4zkbm22bIXzS6pd-26tlEb54BC0kSWXTE35nW9oHIYIlzuo2QyYXFsQ1owg3FiOGrRAXH09mJoH8JqQ3a2gKGJ8yQRR-8Kvk97D22XqhsQmz8ur5Wm_5xaKLZNWCLLGfq_0wm8uHnvNNRZ2mngMn3g6-gaqOwYUjTX2MoEG0jXE8ko5jkNfX41txpD0PW7kJ6UpJ7CoGEdFRKxjwsqIGBgZY3rlOEhxEP_HwfVstTX9VMo4E1k87QnGCSk-5dzGqd71RHOcGtXRmIUltQDrn3Qb5jw0CUPXIU4WnC0YlDFfxyV_RJZSQYDjxhgWDsXh_BErLxgL_RyZ5HkwiLf88YvdQKRBv8oZA_bVX4yEHQ0NNS5Azwsmaaho9jJ2k8pG-UL7w02dQTX2TPJKst-5T6jag8Ap18w8LRKzXaxaDGqw_2BUSGunYYOB30NQ'


url="https://toar-data.fz-juelich.de/api/v2/"
headers = {'AccessToken': token}


files_path = 'files/'
stations_url = url + 'stationmeta/'
timeseries_url = url + 'timeseries/'
timeseries_data_url = url + 'data/timeseries/'

    

# Bounding Box
(min_lat, min_lon, max_lat, max_lon in degrees_north/degrees_east) 

In [83]:
polygon = gpd.read_file('../../data/cdmx/Polygons.gpkg', layer = 'cdmx')
polygon.to_crs(epsg=4326, inplace=True)
bbox = polygon.total_bounds  # minx, miny, maxx, maxy
print('Bounding box:', bbox)
min_lat = bbox[1]
max_lat = bbox[3]
min_lon = bbox[0]
max_lon = bbox[2]

Bounding box: [-99.3649242  19.0487187 -98.9403028  19.5927572]


# Stations metadata

In [84]:
params = {
    'limit': 'None',
    # 'country': 'MX'
    'bounding_box': f'{min_lat},{min_lon},{max_lat},{max_lon}'
}

response = requests.get(stations_url, headers=headers, params=params)

In [85]:
if response.status_code == 200:
    stations_json = response.json()
    
    print(f'Correctly loaded metadata from {len(stations_json)} stations:')
else:
    print(f"Error: {response.status_code} - {response.text}")

Correctly loaded metadata from 42 stations:


In [86]:
stations_data = []
for s in stations_json:
    d = {
        'id': s.get('id'),
        'name': s.get('name'),
        'lat': s.get('coordinates', {}).get('lat',-1),
        'lon': s.get('coordinates', {}).get('lng',-1),
        'alt': s.get('coordinates', {}).get('alt',-1),
        'country': s.get('country'),
        'state': s.get('state'),
        'type': s.get('type', s.get('type_of_area'))
    }
    stations_data.append(d)

# optional: convert to DataFrame for easy inspection
stations_df = pd.DataFrame(stations_data)
stations_df.head(10)

,id,name,lat,lon,alt,country,state,type
0,17,Pedregal,19.32515,-99.2041,2326.0,Mexico,Distrito Federal,traffic
1,1318,Santa Fe,19.35730,-99.2628,-999.0,Mexico,,unknown
2,1634,Tlahuac,19.24640,-99.0100,-999.0,Mexico,,unknown
3,1712,Atizapán,19.57700,-99.2542,-999.0,Mexico,,unknown
4,1873,UAM Xochimilco,19.30440,-99.0738,-999.0,Mexico,,unknown
5,2036,Iztacalco,19.38440,-99.1176,-999.0,Mexico,,unknown
6,2080,Hospital General de,19.41160,-99.1522,-999.0,Mexico,,unknown
7,2217,La Presa,19.53470,-99.1177,-999.0,Mexico,,unknown
8,2313,Xalostoc,19.52590,-99.0824,-999.0,Mexico,,unknown
9,2524,Los Laureles,19.57870,-99.0396,-999.0,Mexico,,unknown


# Time Series

In [99]:
params = {
    'limit': 'None',
    'station_id': ','.join(stations_df['id'].astype('str').tolist()),
    }

response = requests.get(timeseries_url, headers=headers, params=params)

In [100]:
if response.status_code == 200:
    timeseries_json = response.json()
    
    print(f'Correctly loaded metadata from {len(timeseries_json)} timeseries:')
else:
    print(f"Error: {response.status_code} - {response.text}")

Correctly loaded metadata from 786 timeseries:


In [101]:
timeseries_dict = []
for s in timeseries_json:
    d = {
        'timeseries_id': s.get('id'),
        'sampling_frequency': s.get('sampling_frequency'),
        'data_origin': s.get('data_origin'),
        'id': s.get('station', {}).get('id')
    }
    timeseries_dict.append(d)

# optional: convert to DataFrame for easy inspection
timeseries_df = pd.DataFrame(timeseries_dict)
timeseries_df.head(10)

,timeseries_id,sampling_frequency,data_origin,id
0,18,hourly,instrument,17
1,681,irregular data samples of varying length,instrument,1318
2,682,irregular data samples of varying length,instrument,1318
3,683,irregular data samples of varying length,instrument,1318
4,684,irregular data samples of varying length,instrument,1318
5,685,irregular data samples of varying length,instrument,1318
6,686,irregular data samples of varying length,instrument,1318
7,1281,irregular data samples of varying length,instrument,1634
8,1282,irregular data samples of varying length,instrument,1634
9,1283,irregular data samples of varying length,instrument,1634


In [103]:
timeseries_json[-1]

{'id': 410202,
 'label': '',
 'order': 3,
 'sampling_frequency': 'hourly',
 'aggregation': 'none',
 'data_start_date': '2000-01-01T00:00:00+00:00',
 'data_end_date': '2022-12-31T23:00:00+00:00',
 'data_origin': 'ERA5',
 'data_origin_type': 'model',
 'provider_version': 'N/A',
 'sampling_height': -1.0,
 'additional_metadata': {},
 'doi': '',
 'coverage': -1.0,
 'station': {'id': 19262,
  'codes': ['openaq_235230'],
  'name': 'Iztacalco',
  'coordinates': {'lat': 19.384444444444,
   'lng': -99.117777777778,
   'alt': -999.0},
  'coordinate_validation_status': 'not checked',
  'country': 'Mexico',
  'state': '',
  'type': 'unknown',
  'type_of_area': 'unknown',
  'timezone': 'America/Mexico_City',
  'additional_metadata': {'rice_production': 0.0,
   'wheat_production': 41.56590843,
   'soybean_production': 0.0},
  'aux_images': [],
  'aux_docs': [],
  'aux_urls': [],
  'globalmeta': {'mean_topography_srtm_alt_90m_year1994': 2238.0,
   'mean_topography_srtm_alt_1km_year1994': 2237.63824289

In [96]:
timeseries_df.query('sampling_frequency == "hourly"')

,timeseries_id,sampling_frequency,data_origin,id
0,18,hourly,instrument,17
198,94461,hourly,ERA5,17
199,94462,hourly,ERA5,17
200,94463,hourly,ERA5,17
201,94464,hourly,ERA5,17
...,...,...,...,...
781,401136,hourly,ERA5,10196
782,401156,hourly,ERA5,10216
783,401968,hourly,ERA5,11028
784,410201,hourly,ERA5,19261


# Time Series Data

In [16]:
params = {
    'limit': None,
    'format': 'csv',
    }
ts_id = str(timeseries_json[-1].get('id'))
response = requests.get(timeseries_data_url + ts_id, 
                        headers=headers,
                        params=params)

In [25]:
response.content

b'#{\n#    "id": 682,\n#    "label": "",\n#    "order": 9,\n#    "sampling_frequency": "irregular data samples of varying length",\n#    "aggregation": "unknown",\n#    "data_start_date": "2016-05-04T22:00:00+00:00",\n#    "data_end_date": "2022-09-29T21:00:00+00:00",\n#    "data_origin": "instrument",\n#    "data_origin_type": "measurement",\n#    "provider_version": "N/A",\n#    "sampling_height": 2.0,\n#    "additional_metadata": {},\n#    "data_license_accepted": null,\n#    "dataset_approved_by_provider": null,\n#    "doi": "",\n#    "coverage": -1.0,\n#    "station": {\n#        "id": 1318,\n#        "codes": [\n#            "openaq_348"\n#        ],\n#        "name": "Santa Fe",\n#        "coordinates": {\n#            "lat": 19.3573,\n#            "lng": -99.2628,\n#            "alt": -999.0\n#        },\n#        "coordinate_validation_status": "not checked",\n#        "country": "Mexico",\n#        "state": "",\n#        "type": "unknown",\n#        "type_of_area": "unknown",

In [17]:
# save CSV returned in `response` to a file
filename = f"timeseries_{ts_id if 'ts_id' in globals() else (timeseries_meta.get('id') if 'timeseries_meta' in globals() else 'unknown')}.csv"

if response.status_code == 200:
    with open(filename, "wb") as fh:
        fh.write(response.content)
    print(f"Saved CSV to {filename} ({len(response.content)} bytes)")
else:
    print(f"Failed to get CSV: {response.status_code} - {getattr(response, 'text', '')}")

Saved CSV to timeseries_682.csv (1225885 bytes)


In [ ]:
"""
toar_rest_demo: demonstrate use the REST interface to TOAR database v2 from python

@author: s.schroeder, Forschungszentrum Juelich GmbH, Germany (s.schroeder@fz-juelich.de)
@date:   2025-22-07
"""
from io import StringIO
import json
import pandas as pd
import requests

TOAR_SERVICE_URL="https://toar-data.fz-juelich.de/api/v2/"
headers = {'AccessToken': 'eyJraWQiOiI1MTQ0NTM4ODMwNDI3Mjk1NzM2MjI2Mjc4MDM1NTUxMzg3NzY1NDMxNTY4OTIyMjAiLCJ0eXAiOiJhdCtqd3QiLCJhbGciOiJSUzI1NiJ9.eyJzdWIiOiJjNGEyODNjMC00ZjBhLTRjZTYtYjgxMy01ZjY5ZGNlNmU0ZjYiLCJhdWQiOiJ0b2FyX3NlcnZpY2VzIiwic2NvcGUiOiJkaXNwbGF5X25hbWUgb2ZmbGluZV9hY2Nlc3MgZW1haWwgcHJvZmlsZSBlZHVwZXJzb25fc2NvcGVkX2FmZmlsaWF0aW9uIGVkdXBlcnNvbl9lbnRpdGxlbWVudCBlZHVwZXJzb25fdW5pcXVlX2lkIGVkdXBlcnNvbl9hc3N1cmFuY2UiLCJhdXRoX3RpbWUiOjE3NjM3MTQ1NDcsImlzcyI6Imh0dHBzOi8vbG9naW4uaGVsbWhvbHR6LmRlL29hdXRoMiIsImV4cCI6MTc2MzcxODYxNywiaWF0IjoxNzYzNzE0NjE3LCJqdGkiOiI1YzUzMzcwNy0yNTM1LTQzYzItYWUwMS0wZDkwOTExMTNkZmYiLCJjbGllbnRfaWQiOiJ0b2FyX3NlcnZpY2VzIn0.Td3jUPld6jhvabDOstuhGNUu3AM2Hro8Bt1ml3VAcKcETZI88Wbytlv8RDquqVZcPD-kE2NW1TNUSWiyMy4-yWmJAolsXjm2QbDbDWXMvvpegQbBDGDfeRPpXuBBGz7BbYi2XFGrEoz9rHUjSADimJU0mNshMeR72LzOKcknCNxJANymqhJ5aApJY06EfogFXDgZ0QuZm_rVsrTPo1mAGEB0r2T2oP6OPBEcNleklmHNw70a5bLj7V3B4FZ7rnhTzHdzEoKHT9Ni8OeiS2GVcNC8QvrzGuhRDmhssQemXr6tNSbrB_xZN7ri4DbJ9NYxKextMjxjDbud727KFOwADq3f_uansAWj7Unyn6U_G4X0jFPnc-1cs403834ZxA9eZK8BqM6gGtcW4VOyhenCmBhxZA961Co4auKuPklZHoKfyPsHqLu9Ta07nN2-1bP4udrFp_XS99MQEemHcG7-ogRQV3juBIzNqOMSv62PYt0Y7rh6uYeHiVebZUtNhXJz-nd1dpkIzq1EVwbdCirvpWNHgMUlKO2aQ5cKKywITlYEsHS0cH44e9XejI60LcWvRaO5eqrOBxHlDNmBo51rXRX2eGeJ8LeMWqX9LH0eDw7vjHawykLSvSOi-CiUsqoTFNGL13aG0rW6qoyc2x1K79xOncdpS4CqL6Pvvo-6dvU'}

# get data from the REST service
flags = None
timeseries_id=21
if flags:
    result=requests.get(TOAR_SERVICE_URL+f'data/timeseries/{timeseries_id}?flags={flags}&format=csv', headers=headers)
else:
    result=requests.get(TOAR_SERVICE_URL+f'data/timeseries/{timeseries_id}?format=csv', headers=headers)
timeseries_meta = json.loads("\n".join([line[1:] for line in result.text.split('\n') if line.startswith('#')]))
df = pd.read_csv(StringIO(result.text), comment='#', header=1, sep=',',names=["time","value","flags","version","timeseries_id"],parse_dates=["time"],index_col="time",date_parser=lambda x:pd.to_datetime(x,utc=True),infer_datetime_format=True)

# print, what I got
print(f"metadata: {timeseries_meta}")
print(f"data: {df}")

# example of how to access metadata
stationmeta=timeseries_meta["station"]
variable=timeseries_meta["variable"]
data_start_date = min(df.index).strftime("%Y-%m-%d %H:%M:%S+00")
data_end_date = min(df.index).strftime("%Y-%m-%d %H:%M:%S+00")

# show stationmeta data
print(f'retrieved data from station {stationmeta["name"]} for variable {variable["name"]} with timerange {data_start_date} - {data_end_date}')
print(f'station coordinates: {stationmeta["coordinates"]}')

<Response [200]>

In [63]:
string = '''


Missing Value Imputation of Time-Series Air-Quality Data via Deep Neural Networks



'''
print(''.join(word.capitalize().replace(':','').replace('-','').replace(',','').replace('.','') for word in string.split()))

MissingValueImputationOfTimeseriesAirqualityDataViaDeepNeuralNetworks


In [102]:
string = '''




Spatio-temporal Graph Convolutional Neural Network for traffic signal prediction in large-scale urban networks



'''
print(''.join(word.capitalize().replace(':','').replace('-','').replace(',','').replace('.','') for word in string.split()))

SpatiotemporalGraphConvolutionalNeuralNetworkForTrafficSignalPredictionInLargescaleUrbanNetworks
